## Parâmetros necessários

**Comprimento de Bjerrum — `l_B`** <br>
Representa a distância na qual a energia de interação eletrostática entre duas cargas elementares é comparável à energia térmica $k_BT$.

$$
l_B=\frac{e^2}{4\pi\varepsilon_0\varepsilon_r k_BT}
$$
Em outras palavras:
- Se a distância entre duas cargas for menor que $l_B$, a interação eletrostática tende a ser energeticamente importante.
- Se for muito maior que $l_B$, a agitação térmica tende a ser mais relevante.

Para água a 25 °C (298,15 K), temos aproximadamente:
$$
l_B \approx 0,71\ \text{nm}
$$

**Constante dielétrica — `epsilon_r`**<br>
Representa a permissividade relativa do meio, isto é, sua capacidade de responder a um campo elétrico e reduzir as interações eletrostáticas entre cargas. Para a água líquida à temperatura ambiente utilizamos o valor aproximado de 78,5 (adimensional).

**Força iônica — `ionic_strength_M`**<br>
Representa a concentração efetiva de íons dissolvidos na solução. Esse parâmetro é importante porque os íons da solução podem se organizar ao redor das superfícies carregadas, reduzindo o alcance das interações eletrostáticas, fenômeno conhecido como screening eletrostático. Ou seja, ele controla o alcance das interações eletrostáticas entre PPy e MoS₂.

O efeito é representado pelo comprimento de Debye:
$$
\lambda_D = \frac{1}{\kappa}
$$

de forma que quanto maior a força iônica, menor o comprimento de Debye:
$$
I \uparrow
\quad\Rightarrow\quad
\lambda_D \downarrow
$$

Considerando a força iônica:
$$
I=\frac{1}{2}\sum_i c_i z_i^2
$$

Ao substituir as espécies provenientes do APS:
$$
I=\frac{1}{2}
\left[
(2C)(1)^2+C(-2)^2
\right]
$$

Obtemos:
$$
I_{\mathrm{APS}}=3C
$$

Ou seja, se utilizarmos uma concentração de APS de 0,01 mol/L:
$$
I_{\mathrm{APS}}=3(0,01)
$$

$$
I_{\mathrm{APS}}=0,03\ \text{mol/L}
$$

**Densidade de carga inicial — `sigma_inicial_nm2`**<br>
Representa a densidade superficial de carga elétrica da superfície antes da primeira deposição. Para o nosso caso vamos usar:
$$
\sigma_0=-0.00624 \ \text{cargas/nm}^2
$$
Ou seja, estamos considerando que a superfície inicial apresenta carga negativa (equivalente a -0.1 uC/cm²). Isso é importante porque o modelo utiliza a atração eletrostática como critério simplificado para determinar se uma camada pode ser adsorvida.<br>

Com o PPy como positivo, temos:
$$
\sigma_{\mathrm{MoS_2}}<0
$$

$$
\sigma_{\mathrm{PPy}}>0
$$

**Comprimento de segmento — `b_nm_ppy`**<br>
Representa o comprimento característico de um segmento da cadeia polimérica. Ele está relacionado à dimensão da cadeia e aparece na estimativa da espessura da camada adsorvida:
$$
\delta \sim
\left(
\frac{b^2}{f|\sigma|l_B}
\right)^{1/3}
$$

Observe que:
$$
\delta \propto b^{2/3}
$$

Portanto, ao aumentar o comprimento de segmento, a espessura estimada da camada também aumenta.

Da literatura, temos que:

| Grandeza | Valor |
|---|---|
| Comprimento da unidade repetitiva | 0,388 nm (3,88 Å) |
| Comprimento de persistência | 11,2 nm (112 Å) |
| Comprimento de Kuhn | 22,4 nm (224 Å) |

**Grau de polimerização — `N_ppy`**<br>
Representa o número de unidades repetitivas ou segmentos estatísticos considerados na cadeia. No nosso caso vamos considerar $N=32$, ou seja, uma cadeia com 32 unidades. Isso determina o tamanho da cadeia e sua capacidade total de carregar cargas, de forma que, quanto maior $N$, maior a quantidade potencial de grupos carregados por cadeia.

**Fração de monômeros carregados — `f_ppy`**<br>
Corresponde a fração efetiva de segmetos de cadeia carregados (0.32). Aqui deve ser interpretado como fração efetiva de carga por unidade repetitiva, logo, controla diretamente a densidade de carga da cadeia.

**Densidade de cadeias adsorvidas — `chains_adsorbed_nm2`**<br>
Representa a quantidade de cadeias de PPy adsorvidas por unidade de área da superfície ($\Gamma$). Esse parâmetro, combinado com $N$ e $f$, determina a carga depositada pelo polímero:
$$
\sigma_{\mathrm{PPy}}=\Gamma Nf
$$
Obs: A densidade de cadeias adsorvidas foi fornecida previamente. Logo, o modelo não calcula o equilíbrio de adsorção nem determina espontaneamente quantas cadeias conseguem se adsorver.

**Densidade de carga da camada de MoS₂ — `sigma_mos2_layer_nm2`**<br>
Representa a magnitude da contribuição de carga efetiva de cada camada de MoS₂ depositada. O sinal é atribuído automaticamente (PPy: positivo e MoS₂: negativo), assim, quando uma camada de MoS₂ é depositada:
$$
\sigma_{\mathrm{MoS_2}}=-0,3\ \text{cargas/nm}^2
$$

O balanço de cargas passa a ser:
$$
\sigma_{\mathrm{final}}
=
\sigma_{\mathrm{inicial}}+
\sigma_{\mathrm{MoS_2}}
$$
É, obviamente, uma aproximação simplificada! Ele não é calculado a partir da estrutura eletrônica do MoS₂, de seus defeitos ou da concentração de grupos superficiais.

**Espessura da camada de MoS₂ — `espessura_mos2_nm`**<br>
Representa a espessura atribuída a cada nanosheet de MoS₂ depositada (1 nm).

## Implementação 

In [3]:

import numpy as np

from scipy.constants import (
    elementary_charge as e,
    epsilon_0,
    Boltzmann as k_B,
    Avogadro as N_A
)


# Funções auxiliares

def bjerrum_length(T, epsilon_r):
    return e**2 / (
        4 * np.pi * epsilon_0 * epsilon_r * k_B * T
    )


def debye_length(T, epsilon_r, ionic_strength_M):
    kappa_sq = (
        2 * e**2 * N_A * 1000 * ionic_strength_M
        / (epsilon_0 * epsilon_r * k_B * T)
    )
    kappa = np.sqrt(kappa_sq)
    return 1 / kappa, kappa


# MODELO DE JOANNY (1999): ADSORÇÃO E INVERSÃO DE CARGA
def joanny_adsorption_layer(
    T=298.15,
    epsilon_r=78.5,
    ionic_strength_M=0.1899,
    sigma_surface_nm2=-0.00624,
    b_nm=22.4,
    N=32,
    f=0.32,
    chains_adsorbed_nm2=0.01,
    polymer_charge_sign=+1
):
    """
    Estimativa de adsorção de PPy sobre uma superfície carregada.

    Densidades de carga: cargas/nm².
    b: nm.
    N: número de monômeros.
    f: fração de monômeros carregados.
    """

    # Validação
    if T <= 0 or epsilon_r <= 0:
        raise ValueError("T e epsilon_r devem ser positivos.")
    if ionic_strength_M <= 0:
        raise ValueError("A força iônica deve ser positiva.")
    if b_nm <= 0 or N <= 0:
        raise ValueError("b e N devem ser positivos.")
    if not 0 < f <= 1:
        raise ValueError("f deve estar entre 0 e 1.")
    if chains_adsorbed_nm2 < 0:
        raise ValueError("A densidade de cadeias deve ser não negativa.")
    if polymer_charge_sign not in (-1, +1):
        raise ValueError("O sinal do polímero deve ser +1 ou -1.")
    if sigma_surface_nm2 == 0:
        raise ValueError("A superfície precisa ter carga não nula.")

    # Comprimentos eletrostáticos
    l_B = bjerrum_length(T, epsilon_r)

    lambda_D, kappa = debye_length(
        T, epsilon_r, ionic_strength_M
    )

    l_B_nm = l_B * 1e9
    lambda_D_nm = lambda_D * 1e9

    # Verificação da atração eletrostática
    sinal_superficie = np.sign(sigma_surface_nm2)

    if polymer_charge_sign == sinal_superficie:

        return {
            "Adsorção": False,
            "Regime": "Sem atração eletrostática",
            "Carga Superfície (cargas/nm²)": sigma_surface_nm2,
            "Carga do PPy (cargas/nm²)": 0.0,
            "Carga Resultante (cargas/nm²)": sigma_surface_nm2,
            "Carga Excedente (cargas/nm²)": 0.0,
            "Inversão de Carga": False,
            "Espessura da Camada (nm)": 0.0,
            "Comprimento de Debye (nm)": lambda_D_nm,
            "Bjerrum (nm)": l_B_nm,
            "kappa * delta": np.nan
        }

    # Espessura estimada da camada adsorvida
    sigma_abs = abs(sigma_surface_nm2)

    delta_nm = (
        b_nm**2 / (f * sigma_abs * l_B_nm)
    )**(1/3)

    # Critério de screening
    kappa_delta = kappa * delta_nm * 1e-9

    if kappa_delta < 1:
        regime = "Baixo screening"
    else:
        regime = "Screening significativo"

    # Carga fornecida pelo PPy
    sigma_polymer_nm2 = (
        polymer_charge_sign
        * chains_adsorbed_nm2
        * N
        * f
    )

    # Balanço de cargas
    sigma_resultante = (
        sigma_surface_nm2 + sigma_polymer_nm2
    )

    # Identificação da inversão
    inversao = (
        sigma_resultante != 0
        and np.sign(sigma_resultante) != sinal_superficie
    )

    carga_excedente = max(
        0,
        abs(sigma_polymer_nm2) - sigma_abs
    )

    return {
        "Adsorção": True,
        "Regime": regime,
        "Carga Superfície (cargas/nm²)": sigma_surface_nm2,
        "Carga do PPy (cargas/nm²)": sigma_polymer_nm2,
        "Carga Resultante (cargas/nm²)": sigma_resultante,
        "Carga Excedente (cargas/nm²)": carga_excedente,
        "Inversão de Carga": inversao,
        "Espessura da Camada (nm)": delta_nm,
        "Comprimento de Debye (nm)": lambda_D_nm,
        "Bjerrum (nm)": l_B_nm,
        "kappa * delta": kappa_delta
    }


# Modelagem do LbL (PPy / MoS2)
def simular_multicamadas_lbl(
    n_camadas,
    T,
    epsilon_r,
    ionic_strength_M,
    sigma_inicial_nm2,
    b_nm_ppy,
    N_ppy,
    f_ppy,
    chains_adsorbed_nm2,
    sigma_mos2_layer_nm2,
    espessura_mos2_nm
):
    """
    Simulação simplificada da deposição alternada de PPy e MoS2.
    A carga superficial é assinada.
    PPy:
        Carga positiva.
    MoS2:
        Carga negativa.
    A deposição de PPy utiliza a estimativa de adsorção
    de polieletrólito.
    """

    resultados = []

    sigma_atual = sigma_inicial_nm2

    for i in range(n_camadas):
        # Determinação do material
        if i % 2 == 0:
            material = "PPy"
            sinal_material = +1
        else:
            material = "MoS2"
            sinal_material = -1

        sigma_antes = sigma_atual

        # ADSORÇÃO DO PPy

        if material == "PPy":

            res = joanny_adsorption_layer(
                T=T,
                epsilon_r=epsilon_r,
                ionic_strength_M=ionic_strength_M,
                sigma_surface_nm2=sigma_atual,
                b_nm=b_nm_ppy,
                N=N_ppy,
                f=f_ppy,
                chains_adsorbed_nm2=chains_adsorbed_nm2,
                polymer_charge_sign=sinal_material
            )

            sigma_nova = res["Carga Resultante (cargas/nm²)"]
            espessura = res["Espessura da Camada (nm)"]
            regime = res["Regime"]
            adsorcao = res["Adsorção"]
            sigma_depositada = res["Carga do PPy (cargas/nm²)"]

        # ADSORÇÃO DE MoS2

        else:
            # Verifica se existe atração eletrostática
            if np.sign(sigma_atual) != sinal_material:

                # Contribuição efetiva de carga do MoS2
                sigma_depositada = (
                    sinal_material * sigma_mos2_layer_nm2
                )

                sigma_nova = sigma_atual + sigma_depositada
                espessura = espessura_mos2_nm
                regime = "Adsorção de nanosheet (simplificada)"
                adsorcao = True
            else:

                sigma_depositada = 0.0
                sigma_nova = sigma_atual
                espessura = 0.0
                regime = "Sem atração eletrostática"
                adsorcao = False

        # Armazenamento dos resultados

        resultados.append({

            "Camada": i + 1,
            "Material": material,
            "Adsorção": adsorcao,
            "Carga Antes (cargas/nm²)": sigma_antes,
            "Carga Depositada (cargas/nm²)": sigma_depositada,
            "Carga Após (cargas/nm²)": sigma_nova,
            "Espessura (nm)": espessura,
            "Regime": regime
        })

        # atualiza a superfície
        sigma_atual = sigma_nova

    return resultados


# Execução
if __name__ == "__main__":

    # PARÂMETROS DO SISTEMA
    params = {
        "T": 298.15,
        "epsilon_r": 78.5,
        "ionic_strength_M": 0.1899,
        "sigma_inicial_nm2": -0.00624,
        "b_nm_ppy": 22.4,
        "N_ppy": 32,
        "f_ppy": 0.32,
        "chains_adsorbed_nm2": 0.01,
        "sigma_mos2_layer_nm2": 0.3,
        "espessura_mos2_nm": 1.0
    }

    # CAMADA ÚNICA: PPy SOBRE MoS2
    resultado_unico = joanny_adsorption_layer(
        T=params["T"],
        epsilon_r=params["epsilon_r"],
        ionic_strength_M=params["ionic_strength_M"],
        sigma_surface_nm2=params["sigma_inicial_nm2"],
        b_nm=params["b_nm_ppy"],
        N=params["N_ppy"],
        f=params["f_ppy"],
        chains_adsorbed_nm2=params["chains_adsorbed_nm2"],
        polymer_charge_sign=+1
    )

    camadas = simular_multicamadas_lbl(
        n_camadas=30,
        **params
    )

    # ESPESSURA TOTAL DO FILME
    espessura_total = sum(
        c["Espessura (nm)"] for c in camadas
    )

import pandas as pd

df_resultados = pd.DataFrame(camadas)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

display(df_resultados)

with open("resultados_simulacao.txt", "w", encoding="utf-8") as arquivo:
    arquivo.write(df_resultados.to_string(index=False))

,Camada,Material,Adsorção,Carga Antes (cargas/nm²),Carga Depositada (cargas/nm²),Carga Após (cargas/nm²),Espessura (nm),Regime
0,1,PPy,True,-0.00624,0.1024,0.09616,70.603952,Screening significativo
1,2,MoS2,True,0.09616,-0.3000,-0.20384,1.000000,Adsorção de nanosheet (simplificada)
2,3,PPy,True,-0.20384,0.1024,-0.10144,22.086526,Screening significativo
3,4,MoS2,False,-0.10144,0.0000,-0.10144,0.000000,Sem atração eletrostática
4,5,PPy,True,-0.10144,0.1024,0.00096,27.871101,Screening significativo
5,6,MoS2,True,0.00096,-0.3000,-0.29904,1.000000,Adsorção de nanosheet (simplificada)
6,7,PPy,True,-0.29904,0.1024,-0.19664,19.437815,Screening significativo
7,8,MoS2,False,-0.19664,0.0000,-0.19664,0.000000,Sem atração eletrostática
8,9,PPy,True,-0.19664,0.1024,-0.09424,22.352868,Screening significativo
9,10,MoS2,False,-0.09424,0.0000,-0.09424,0.000000,Sem atração eletrostática
